In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src import skillcorner
from src.load import load_match, load_meta, list_matches

SC_MATCHES = [m for m in list_matches() if not m.startswith("metrica")]
RESTART_TAKER = {"throw_in_for", "goal_kick_for", "corner_for"}   # offside doesn't apply to these


def det(rows):
    """True only if there is exactly one row and its position was detected."""
    return len(rows) == 1 and bool(rows.is_detected.fillna(False).iloc[0])


def x_of(rows):
    """x position (attacking frame) of a single row, else NaN."""
    return rows.x_att.iloc[0] if len(rows) == 1 else np.nan


print(len(SC_MATCHES), "SkillCorner matches")

20 SkillCorner matches


In [2]:
out = []
for m in SC_MATCHES:
    home_id = load_meta(m)["teams"]["home"]["team_id"]
    ev = skillcorner.load_dynamic_events(m)
    # a pass = a player possession that ended in a pass; frame_end = the frame the ball is played
    passes = ev[(ev.event_type == "player_possession") & (ev.end_type == "pass")]
    df = load_match(m, columns=["frame_id", "object_type", "player_id", "team_id", "x_home_m", "is_detected"])
    frames = dict(tuple(df[df.frame_id.isin(passes.frame_end)].groupby("frame_id")))

    for p in passes.itertuples():
        f = frames[p.frame_end].copy()
        # attacking frame: the PASSER's team attacks towards +x
        f["x_att"] = f.x_home_m * (1 if str(p.team_id) == home_id else -1)

        pl = f[f.object_type == "player"]
        ball = f[f.object_type == "ball"]
        passer = pl[pl.player_id == str(p.player_id)]
        receiver = (pl[pl.player_id == str(int(p.player_targeted_id))]
                    if pd.notna(p.player_targeted_id) else pl.iloc[:0])

        # defenders = the other team; they defend the +x goal, so the deepest has the LARGEST x_att.
        # Ties broken by player_id so the result is the same on every machine.
        defs = pl[pl.team_id != str(p.team_id)].dropna(subset=["x_att"])
        defs = defs.sort_values(["x_att", "player_id"], ascending=[False, True], kind="stable")
        second_last = defs.iloc[1:2]

        out.append(dict(
            match=m, event_id=p.event_id, frame=p.frame_end, period=p.period,
            tracked=len(pl) > 0,
            passer=det(passer), receiver=det(receiver), ball=det(ball), second_last_def=det(second_last),
            restart=(p.start_type == "unknown") and (p.game_interruption_before in RESTART_TAKER),
            x_receiver=x_of(receiver), x_ball=x_of(ball), x_second_last=x_of(second_last),
        ))

res = pd.DataFrame(out)
res["is_usable"] = res[["passer", "receiver", "ball", "second_last_def"]].all(axis=1)
res["offside_possible"] = (~res.restart) & (res.x_receiver > 0) & (res.x_receiver > res.x_ball)
res["offside"] = res.offside_possible & (res.x_receiver > res.x_second_last)
print(f"passes checked: {len(res):,}")

passes checked: 16,072


In [3]:
print(f"all passes: {len(res):,}   usable: {res.is_usable.sum():,} ({res.is_usable.mean():.1%})")
display(res[["tracked", "passer", "receiver", "ball", "second_last_def", "is_usable"]].mean().map("{:.1%}".format))
display(res.groupby("match").is_usable.agg(total="size", usable="sum", share="mean").sort_values("share"))

all passes: 16,072   usable: 6,349 (39.5%)


tracked            100.0%
passer              96.9%
receiver            81.4%
ball                98.0%
second_last_def     45.2%
is_usable           39.5%
dtype: str

,total,usable,share
match,,,
2016236,823,205,0.249089
2010085,836,256,0.306220
2015213,736,231,0.313859
2007448,752,239,0.317819
2017461,654,214,0.327217
2011166,630,210,0.333333
1925299,912,309,0.338816
2013725,766,261,0.340731
2006363,845,289,0.342012


## 6. Offside-relevant passes

In [4]:
rel = res[res.offside_possible]
usable_rel = rel[rel.is_usable]
print(f"offside-possible:      {len(rel):,}   ({len(rel)/len(res):.1%} of all passes)")
print(f"  of which usable:     {len(usable_rel):,}   ({rel.is_usable.mean():.1%})")
print(f"  usable AND offside:  {usable_rel.offside.sum():,}   ({usable_rel.offside.mean():.1%} of usable)")
display(rel[["passer", "receiver", "ball", "second_last_def", "is_usable"]].mean().map("{:.1%}".format))

res.to_parquet(ROOT / "data" / "processed" / "qa" / "passes_skillcorner.parquet", index=False)

offside-possible:      5,279   (32.8% of all passes)
  of which usable:     3,120   (59.1%)
  usable AND offside:  85   (2.7% of usable)


passer             97.7%
receiver           82.3%
ball               98.6%
second_last_def    62.4%
is_usable          59.1%
dtype: str

In [5]:
rel = res[res.offside_possible].copy()

# 1) Usable share by where the ball is (attacking frame: the passer's team attacks +x)
rel["ball_zone"] = pd.cut(rel.x_ball, [-60, -17.5, 17.5, 60],
                          labels=["own third", "middle third", "final third"])
display(rel.groupby("ball_zone", observed=True).is_usable
          .agg(passes="size", usable_share="mean").style.format({"usable_share": "{:.1%}"}))

# 2) Usable share by how far ahead of the ball the receiver is
rel["receiver_ahead"] = pd.cut(rel.x_receiver - rel.x_ball, [0, 10, 20, 30, 100],
                               labels=["0-10 m", "10-20 m", "20-30 m", "30+ m"])
display(rel.groupby("receiver_ahead", observed=True).is_usable
          .agg(passes="size", usable_share="mean").style.format({"usable_share": "{:.1%}"}))

# 3) How close are the calls? margin = receiver x - second-last defender x (positive = offside)
u = rel[rel.is_usable].copy()
u["margin_m"] = u.x_receiver - u.x_second_last
display(pd.cut(u.margin_m, [-100, -5, -2, -1, -0.5, 0, 0.5, 1, 2, 5, 100])
          .value_counts().sort_index().rename("usable passes").to_frame())

,passes,usable_share
ball_zone,,
own third,369,7.9%
middle third,3333,52.1%
final third,1577,85.9%


,passes,usable_share
receiver_ahead,,
0-10 m,2957,75.0%
10-20 m,1584,48.9%
20-30 m,488,24.6%
30+ m,250,3.2%


,usable passes
margin_m,
"(-100.0, -5.0]",1693
"(-5.0, -2.0]",856
"(-2.0, -1.0]",296
"(-1.0, -0.5]",112
"(-0.5, 0.0]",78
"(0.0, 0.5]",43
"(0.5, 1.0]",22
"(1.0, 2.0]",19
"(2.0, 5.0]",1


## 8. Metrica offside-relevant passes

In [6]:
import json

RESTART_SUBTYPES = {"THROW IN", "CORNER KICK", "GOAL KICK"}   # offside doesn't apply to these


def metrica_passes(game):
    """One row per PASS: frame, passer, receiver, team side, and whether it's a restart."""
    if game in (1, 2):
        ev = pd.read_csv(ROOT / f"data/raw/metrica_sample_data/data/Sample_Game_{game}/Sample_Game_{game}_RawEventsData.csv")
        ev = ev.rename(columns={"Type": "type", "Subtype": "subtype", "Start Frame": "frame",
                                "From": "passer", "To": "receiver", "Team": "team"})
        ev["team_side"] = ev.team.str.lower()
    else:
        d = json.loads((ROOT / "data/raw/metrica_sample_data/data/Sample_Game_3/Sample_Game_3_events.json").read_text())["data"]
        home_id = load_meta("metrica_game_3")["teams"]["home"]["team_id"]
        rows = []
        for e in d:
            st = e.get("subtypes") or []
            st = [st] if isinstance(st, dict) else st
            rows.append(dict(type=e["type"]["name"], subtype=",".join(s["name"] for s in st),
                             frame=e["start"]["frame"], passer=(e.get("from") or {}).get("id"),
                             receiver=(e.get("to") or {}).get("id"),
                             team_side="home" if e["team"]["id"] == home_id else "away"))
        ev = pd.DataFrame(rows)
    ev["subtype"] = ev.subtype.fillna("")
    # a restart pass = a SET PIECE (throw-in / corner / goal kick) by the same player on the same frame
    restarts = ev[(ev.type == "SET PIECE") & ev.subtype.isin(RESTART_SUBTYPES)]
    restart_keys = set(zip(restarts.frame, restarts.passer))
    passes = ev[ev.type == "PASS"].copy()
    passes["restart"] = [((f, p) in restart_keys) or ("GOAL KICK" in s)
                         for f, p, s in zip(passes.frame, passes.passer, passes.subtype)]
    return passes


mout = []
for g in (1, 2, 3):
    m = f"metrica_game_{g}"
    passes = metrica_passes(g)
    df = load_match(m, columns=["frame_id", "object_type", "player_id", "team_side", "x_home_m", "is_detected"])
    frames = dict(tuple(df[df.frame_id.isin(passes.frame)].groupby("frame_id")))

    for p in passes.itertuples():
        f = frames[p.frame].copy()
        f["x_att"] = f.x_home_m * (1 if p.team_side == "home" else -1)   # passer's team attacks +x
        pl = f[(f.object_type == "player") & f.x_att.notna()]             # Metrica: NaN = not on pitch
        ball = f[f.object_type == "ball"]
        passer = pl[pl.player_id == p.passer]
        receiver = pl[pl.player_id == p.receiver] if pd.notna(p.receiver) else pl.iloc[:0]
        defs = pl[pl.team_side != p.team_side]
        defs = defs.sort_values(["x_att", "player_id"], ascending=[False, True], kind="stable")
        second_last = defs.iloc[1:2]
        mout.append(dict(
            match=m, frame=p.frame, restart=p.restart,
            passer=det(passer), receiver=det(receiver), ball=det(ball), second_last_def=det(second_last),
            x_receiver=x_of(receiver), x_ball=x_of(ball), x_second_last=x_of(second_last),
        ))

mres = pd.DataFrame(mout)
mres["is_usable"] = mres[["passer", "receiver", "ball", "second_last_def"]].all(axis=1)
mres["offside_possible"] = (~mres.restart) & (mres.x_receiver > 0) & (mres.x_receiver > mres.x_ball)
mres["offside"] = mres.offside_possible & (mres.x_receiver > mres.x_second_last)
mres["margin_m"] = mres.x_receiver - mres.x_second_last

mrel = mres[mres.offside_possible & mres.is_usable]
print(f"Metrica passes: {len(mres):,}   usable: {mres.is_usable.sum():,}   restarts: {mres.restart.sum()}")
print(f"offside-possible & usable: {len(mrel):,}   offside: {mrel.offside.sum()} ({mrel.offside.mean():.1%})")
display(mrel.groupby("match").offside.agg(offside_possible="size", offside="sum"))
display(pd.cut(mrel.margin_m, [-100, -5, -2, -1, -0.5, 0, 0.5, 1, 2, 5, 100])
          .value_counts().sort_index().rename("Metrica usable passes").to_frame())
mres.to_parquet(ROOT / "data" / "processed" / "qa" / "passes_metrica.parquet", index=False)

Metrica passes: 2,884   usable: 2,884   restarts: 141
offside-possible & usable: 861   offside: 11 (1.3%)


,offside_possible,offside
match,,
metrica_game_1,228,2
metrica_game_2,310,0
metrica_game_3,323,9


,Metrica usable passes
margin_m,
"(-100.0, -5.0]",649
"(-5.0, -2.0]",136
"(-2.0, -1.0]",40
"(-1.0, -0.5]",9
"(-0.5, 0.0]",16
"(0.0, 0.5]",7
"(0.5, 1.0]",3
"(1.0, 2.0]",1
"(2.0, 5.0]",0


In [7]:
# Combine both sources: usable, offside-possible passes only
sc = res[res.offside_possible & res.is_usable].assign(source="skillcorner")
me = mres[mres.offside_possible & mres.is_usable].assign(source="metrica")
cols = ["source", "match", "frame", "x_receiver", "x_ball", "x_second_last", "offside"]
pool = pd.concat([sc[cols], me[cols]], ignore_index=True)
pool["margin_m"] = pool.x_receiver - pool.x_second_last          # > 0 = offside
pool["abs_margin_m"] = pool.margin_m.abs()
BINS = [0, 0.5, 1, 2, 5]
pool["margin_bin"] = pd.cut(pool.abs_margin_m, BINS, include_lowest=True)

# Balanced selection: in each |margin| bin, keep every offside case and an equal number of
# onside cases drawn at random (fixed seed). Difficulty (margin) is then independent of the answer.
rng_seed = 2026
pos = pool[pool.offside]
parts = [pos]
for b, n_pos in pos.groupby("margin_bin", observed=True).size().items():
    neg_bin = pool[(~pool.offside) & (pool.margin_bin == b)]
    parts.append(neg_bin.sample(n=min(n_pos, len(neg_bin)), random_state=rng_seed))
offside_set = pd.concat(parts, ignore_index=True).sort_values(["source", "match", "frame"]).reset_index(drop=True)

print(f"offside candidate set: {len(offside_set)} items "
      f"({offside_set.offside.sum()} offside / {(~offside_set.offside).sum()} onside)")
display(pd.crosstab(offside_set.margin_bin, offside_set.offside.map({True: "offside", False: "onside"})))
display(pd.crosstab(offside_set.source, offside_set.offside.map({True: "offside", False: "onside"})))
offside_set.assign(margin_bin=offside_set.margin_bin.astype(str)).to_parquet(
    ROOT / "data" / "processed" / "qa" / "offside_candidates.parquet", index=False)

offside candidate set: 192 items (96 offside / 96 onside)


offside,offside,onside
margin_bin,,
"(-0.001, 0.5]",50,50
"(0.5, 1.0]",25,25
"(1.0, 2.0]",20,20
"(2.0, 5.0]",1,1


offside,offside,onside
source,,
metrica,11,16
skillcorner,85,80
